# Artefato 1: Entendimento do Problema & Dados (EDA + Pipeline)

## Mapeamento de Stakeholders e Necessidades

Para que o projeto **SpectraAI** saia do campo teórico e das ideias, e realmente gere impacto na operação da Frontera Minerals, deve-se identificar os *stakeholders* e suas expectativas. Segundo [Freeman (1984)](#ref-freeman), o resultado de um projeto está diretamente ligado à capacidade de atender aos diferentes interesses envolvidos. Nesse sentido, isso significa que o modelo de rede neural não deve apenas "funcionar", mas sim entregar informações em formatos que façam sentido tanto para quem está no campo quanto para quem decide os investimentos.

Entende-se, portanto, que essa conexão entre a tecnologia e o valor de negócio acontece quando gera-se, por meio das bandas espectrais do sensor ASTER, dados acionáveis. Com isso, uma vez que as expectativas acerca dessa ferramenta varia conforme o perfil dos envolvidos, para garantir que o **SpectraAI** cumpra seu papel, mapeia-se na tabela abaixo as necessidades específicas de cada grupo, o nível de detalhamento exigido e as decisões que os resultados irão suportar:

**Tabela 1: Mapeamento de Stakeholders e Necessidades**

| Stakeholder | Papel no Projeto | Necessidades de Informação | Nível de Detalhe | Decisões Suportadas |
| --- | --- | --- | --- | --- |
| **Geologia (Frontera)** | Especialista | Localização de áreas potenciais para extração de minérios. | **Alto:** Mapas de probabilidade e coordenadas dos alvos. | Priorização de áreas para campanhas de campo e sondagens. |
| **Diretoria (Frontera)** | Patrocinador | Retorno sobre investimento e redução de riscos operacionais. | **Baixo:** Relatórios de performance e economia de tempo/recurso. | Alocação de orçamento e decisão de novos requerimentos minerais por meio do ranking de geolocalizações. |
| **Equipe Docente (Inteli)** | Avaliador | Rigor metodológico e eficácia do pipeline de Deep Learning. | **Alto:** Métricas de treino, estruturação da documentação, código e lógica aplicada. | Validação das competências técnicas e notas do grupo. |
| **Alunos (Grupo 1)** | Desenvolvedores | Qualidade do dataset e estabilidade da rede neural durante o treino. | **Máximo:** Logs de erro, hiperparâmetros e matrizes de confusão. | Ajustes na arquitetura do modelo e estratégias de pré-processamento. |

*Fonte: Elaboração própria*

Dessa forma, o sucesso do **SpectraAI** é definido pela convergência dessas necessidades. Sendo assim, não basta que o modelo atinja métricas de precisão elevadas se os resultados não forem interpretáveis para a equipe de Geologia ou se não oferecerem segurança para as decisões de investimento da Diretoria. Portanto, a arquitetura do pipeline de dados deve estar preparada para por meio da técnica do Deep Learning entregar uma ferramenta que reduza a incerteza exploratória, transformando imagens de satélite em alvos minerais estratégicos.

## Escolha das Métricas e Justificativa

&emsp;Para avaliar o desempenho do modelo, o problema será tratado como uma tarefa de classificação binária, em que cada tile de imagem representa uma área com maior ou menor potencial de ocorrência de terras raras. Por existir um leve desbalanceamento entre as classes(63% áreas positivas e 37% negativas), a avaliação não pode se basear apenas em acurácia, que tende gerar interpretações enganosas em dados assim. Por isso, a análise prioriza métricas que consideram melhor o equilíbrio entre erros e acertos. A seguir, apresentamos as fórmulas de cada métrica:
$$
Acurácia = \frac{TP + TN}{TP + TN + FP + FN}
$$
&emsp;A acurácia fornece uma medida geral da capacidade do modelo de fazer previsões corretas em todas as classes; O Recall mede a proporção de exemplos positivos que o modelo conseguiu identificar corretamente em relação ao número total de exemplos positivos e Precision mede a proporção de exemplos que o modelo previu como positivos e que realmente eram positivos. A partir dessas métricas, definem-se as métricas adotadas no projeto: F1-score e ROC-AUC.

$$
Precision = \frac{TP}{TP + FP}
$$

$$
Recall = \frac{TP}{TP + FN}
$$
&emsp;O F1-score é importante por combinar precisão e recall em um único valor, ajudando a medir o equilíbrio entre evitar falsos positivos e não deixar passar áreas potencialmente relevantes. Já a ROC-AUC permite observar a capacidade geral do modelo em separar as duas classes independentemente de um limiar fixo de decisão. A acurácia pode ser apresentada apenas como métrica complementar, servindo como referência geral, mas não como critério decisivo.

$$
F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}
$$

$$ AUC = \int_{0}^{1} TPR(FPR^{-1}(x)) \, dx $$


&emsp; Segundo Sokolova e Lapalme (2009), Se o desbalanceamento ocorre porque a classe de interesse (positiva) é pequena e a classe negativa é grande, deve-se usar o F-score (e suas componentes Precision e Recall). Dessa forma, como critério de aceitação, o modelo será considerado satisfatório caso apresente ROC-AUC ≥ 0,70 e F1-score ≥ 0,60 no conjunto de teste, mantendo recall ≥ 0,65, de forma a não negligenciar áreas potencialmente relevantes. A acurácia pode ser observada como indicador complementar, com expectativa em torno de ≥ 0,70, mas com menor relevância.

## Descrição do sensor ASTER

&emsp;Ao longo de duas décadas, o sensor ASTER (Advanced Spaceborne Thermal Emission and Reflection Radiometer) mudou muito a cartografia geológica e a exploração mineral. a transição de um sistema antigo, o Landsat, para o ASTER permitiu a identificação precisa de minerais de alteração, como argilas e carbonatos, graças às suas bandas exclusivas nos espectros SWIR (infravermelho de ondas curtas) e TIR (infravermelho térmico).

&emsp;Segundo o manual, ASTER User Handbook, o ASTER consiste em três subsistemas diferentes : o Visível e Infravermelho Próximo (VNIR)
possui três bandas com resolução espacial de 15 m e um telescópio adicional para visualização traseiraestéreo; o Infravermelho de Ondas Curtas (SWIR) possui 6 bandas com resolução espacial de 30 m; e o Infravermelho Térmico (TIR) possui 5 bandas com resolução espacial de 90 m. Cada subsistema opera em uma região espectral diferente, com seu(s) próprio(s) telescópio(s), e é construído por uma empresa japonesa diferente.” (ABRAMS; HOOK; RAMACHANDRAN, 2002, p. 9). As faixas espectrais são mostradas a seguir:
<div align="center">
  <strong>Figura 1</strong> – Faixas espectrais do VNIR, SWIR e TIR
  <br>
  <img src="../../assets/bandas.png" alt="Figura 1 – Faixas espectrais do VNIR, SWIR e TIR" style="max-width: 80%;">
  <p style="font-size: 0.9em;">Fonte: Aster User Handbook v2.</p>
</div>


&emsp;O subsistema VNIR (visible and near-infrared) opera com três bandas espectrais (1 a 3) e possui capacidade estereoscópica na Banda 3, permitindo a geração de Modelos Digitais de Elevação (DEM) com precisão vertical de cerca de 15 metros, essenciais para análises topográficas e estruturais. Na aplicação mineral, serve para mapear a presença de óxidos de ferro (férricos e ferrosos), permitindo a detecção de zonas ricas em ferro, frequentemente associadas a rochas máficas ou mineralizações de sulfetos intemperizados

&emsp;O short wave infrared (SWIR) oferece seis bandas (Bandas 4 a 9) estreitas, responsáveis pela discriminação mineralógica específica, isto é, mapeamento de zonas de alteração hidrotermal, que, por sua vez, ajuda a distinguir minerais do grupo Al-OH (como caulinita, ilita e muscovita), minerais Mg-OH (clorita, epidoto), carbonatos e sulfatos (como alunita). Essa característica é usada para identificar zonas de alteração (argílica, fílica, propilítica) em depósitos de pórfiro cuprífero e sistemas epitermais de ouro.

&emsp;Por último, o thermal infrared (TIR), com cinco bandas (Bandas 10 a 14) , permite a derivação da emissividade espectral e temperatura da superfície, ajudando a estimar o conteúdo de sílica (SiO2), distinguindo rochas ígneas félsicas de máficas/ultramáficas e mapeando a abundância de quartzo em rochas sedimentares. Além disso, o TIR também serve para identificar carbonatos e zonas de silicificação através de índices minerais específicos (como os Índices de Quartzo, Carbonato e Máfico de Ninomiya), complementando o mapeamento de alteração onde o SWIR pode ser limitado.

## 2. Inventário dos Dados Disponibilizados

### 2.1 Introdução

&emsp;&emsp;Este documento apresenta o inventário completo dos dados disponibilizados pelo parceiro de projeto **Frontera Minerals** para o desenvolvimento do sistema de *deep learning* aplicado à visão computacional para identificação de áreas com potencial de presença de Terras Raras. O projeto tem como objetivo a análise de imagens de satélite (ASTER e Landsat) para detecção de assinaturas espectrais características de minerais de terras raras.

&emsp;&emsp;Os dados foram compartilhados via Dropbox e incluem imagens multiespectrais do satélite ASTER (Advanced Spaceborne Thermal Emission and Reflection Radiometer), coordenadas georreferenciadas de terras raras conhecidas, e documentação técnica de apoio.

**Link de acesso aos dados do parceiro:**  
[Dropbox](https://www.dropbox.com/scl/fo/5w7xvlpanxdeu5lgt4a49/AOMKwMTI2Z3B5E7wzfs1ysw?rlkey=aolxn05zq4aqw09wz68297wa0&st=y4zevnfr&dl=0)

### 2.2 Estrutura de Diretórios (Dropbox)

&emsp;&emsp;A estrutura de diretórios dos dados disponibilizados pelo parceiro no Dropbox está organizada da seguinte forma:

```
Inteli/
├── Coordenadas Minas com Terras Raras/
│   ├── ASTER/
│   │   ├── AST_L1T_004-20251216_145940/
│   │   │   ├── AST_L1T_*_VNIR.tif
│   │   │   ├── AST_L1T_*_VNIR_B01.tif
│   │   │   ├── AST_L1T_*_VNIR_B02.tif
│   │   │   ├── AST_L1T_*_VNIR_B03N.tif
│   │   │   ├── AST_L1T_*_TIR.tif
│   │   │   └── AST_L1T_*_TIR_B10-B14.tif
│   │   └── AST_L1T_004-20251216_152230/
│   │       ├── (mesma estrutura de bandas)
│   │       └── AST_L1T_*_SWIR_B04.tif
│   ├── Coordenadas MINA Serra Verde.xlsx
│   └── Coordenadas MINA CBMM.xlsx
├── Banco de Dados Positivo-Negativo.xlsx
├── Passo a Passo Aster.docx
├── Passo a Passo Aster.pdf
└── Q-A Imagens Landsat e Aster (1).docx
```

### REFERÊNCIAS

ABRAMS, Michael; HOOK, Simon; RAMACHANDRAN, Bhaskar. **ASTER User Handbook**. Version 2, 2002. p.9. Disponível em https://asterweb.jpl.nasa.gov/content/03_data/04_Documents/aster_user_guide_v2.pdf. Acesso em 3 fev. 2026

ABRAMS, Michael; YAMAGUCHU, Yasushi. **Twenty Years of ASTER Contributions to Lithologic Mapping and Mineral Exploration**, 2019. Disponível em: https://doi.org/10.3390/rs11111394 Acesso em 3 fev. 2026

SOKOLOVA, Marina; LAPALME, Guy. **A systematic analysis of performance measures for classification tasks. Information Processing and Management**, v. 45, n. 4, p. 435, 2009.

FREEMAN, R. Edward. Strategic Management: A Stakeholder Approach. Boston: Pitman, 1984. <a id="ref-freeman"></a>